# Práctica 2: Modelo cinemático inverso de un manipulador y planteamiento de trayectoria


## Objetivo



El objetivo de esta práctica es que el alumno comprenda, interprete y modifique el modelo de cinemática directa e inversa d eun moanipulador serial.


### Metas 

- Que el alumno aplique un modelo cinemático, obteniendo las matrices de transformación y la postura del manipulador 
- Que el alumno aplique un modelo cinemático inverso para calcular una trayectoria a partir de una posición actual hacia una posición final
- Que el alumno grafique y analice los resultados del modelo

### Contribución al perfil del egresado

La siguiente práctica contribuye en los siguientes puntos al perfil del egresado:

#### Aptitudes y habilidades

- Para modelar, simular e interpretar el comportamiento de los sistemas mecatrónicos.
- Para diseñar, construir, operar y mantener los sistemas mecatrónicos y sus componentes.

#### Actitudes

- Tener confianza en su preparación académica.
- Comprometido con su actualización, superación y competencia profesional.

#### De tipo social

- Promover el cambio en la mentalidad frente a la competitividad internacional.



## Rúbrica de evaluación



La evaluación de la práctica contará de los siguientes puntos y se evaluará con los siguientes criterios:

| Elemento | Porcentaje |
| ------:| -----------:|
| **Cuestionario previo** | 15% | 
| **Desarrollo** | 35% |
| **Análisis de resultados**  | 35% |
| **Conclusiones** | 15% |

<br>


| Elemento | Malo | Regular | Bueno |
| ------:| ------ | --------| ------|
| **Cuestionario previo** | El trabajo no contiene cuestionario previo o todas las preguntas son incorrectas (0%)| Al menos la mitad de las preguntas son correctas (8%) |  Todas las preguntas son correctas (15%) |
| **Desarrollo** | El trabajo no contiene desarrollo o su planteamiento no concuerda con lo deseado (0%) | El desarrollo está mal planteado o no llega a los resultados esperados (10%) | El desarrollo tiene un planteamiento adecuado y llega a los resultados esperados (35%) |
| **Análisis de resultados**  | El trabajo no contiene análisis de resultados o la información no se está interpretando correctamente (0%) | La interpretación de los resultados es parcial o desorganizada (10%) | Realiza un correcto análisis de los resultados de forma organizada   (35%) |
| **Conclusiones** | El trabajo no contiene conclusiones o no hacen referencia al trabajo desarrollado y los objetivos planteados (0%) | La redacción de las conclusiones es desorganizada o confusa (8%) | Las conclusiones del trabajo son claras y hacen referencia al trabajo desarrollado y los objetivos planteados (15%) | 



## Introducción

### Transformaciones homogéneas
Las transformaciones homogéneas permiten hacer el planteamiento del modelo cinemático de un robot, considerando las posiciones y orientaciones de las juntas del robot respecto al sistema de referencia de una junta anterior

Este planteamiento es el **modelo de cinemática directa**, que nos permite obtener la posición y velocidad del efector final de un manipulador en términos de los valores de la posición y velocidad de sus juntas (espacio de trabajo)

A través de este modelo se puede obtener el **modelo de cinemática inversa**, que permite obtener la velocidad de las juntas de un robot a partir de la velocidad deseada del efector final.

### Planteamiento de una trayectoria
Si se conoce el punto inicial y final de una trayectoria deseada, se pueden obtener los puntos intermedios de la trayectoria. La forma más fácil de realizar esta interpolación es a través de un spline. El orden del spline permitirá controlar las condiciones inicial y final de la posición, velocidad ó aceleración que tendrá el efector final durante el trayecto. 

Juntando la interpolación de la trayectoria y el modelo de la cinemática inversa, se pueden obtener todos los puntos intermedios de la trayectoria que deben seguir las juntas del robot para que el efector final siga una trayectoria.

### Inyección de dependencias en python
Al crear una clase, es posible agregar atributos y métodos a la misma, que se verán reflejados al instanciar un objeto, por ejemplo:

In [1]:
# Se define una clase
class ClaseEjemplo():
  def metodo_original(self):
    print("Metodo original")
# Fuera de la clase, se define un método nuevo
def metodo_inyectado(self):
  print("Método inyectado en la clase")
# Se asigna el método dentro de la clase
ClaseEjemplo.nuevo_metodo = metodo_inyectado
# Al instanciar el objeto, éste contiene el método inyectado
objeto = ClaseEjemplo()
objeto.nuevo_metodo()

Método inyectado en la clase


## Cuestionario previo



Responder de forma breve las siguientes preguntas:

- ¿Que son las transformaciones homogéneas?
>Son matrices matemáticas que permiten representar rotaciones y traslaciones de un sistema de coordenadas en un solo modelo.

- ¿Que nos permite obtener el modelo de cinemática directa de un manipulador?
>Permite calcular la posición y orientación del efector final a partir de los valores de las articulaciones del robot.

- ¿Que nos permite obtener el modelo de cinemática inversa de un manipulador?
>Permite calcular los valores de las articulaciones necesarios para alcanzar una posición y orientación deseada del efector final.

- ¿De que formas se puede interpolar la trayectoria de un efector final entre dos puntos?
>Se puede interpolar mediante trayectorias lineales, circulares o polinomiales en el espacio articular o cartesiano.

## Desarrollo



### 1. Planteamiento de la cinemática directa
En esta primera parte, se crearán las transformaciones homogéneas y el modelo de cinemática directa de un robot RRR, incluyendo la matriz del Jacobiano. Se recomienda usar **Sympy** para el planteamiento de las expresiones. 
Un diagrama del robot se muestra en la imagen:

<img src="imagenes/p2_1.png" alt = "Robot RRR" width="300" height="300" display= "block"/>

** Considerar valores cualesquiera para las dimensiones de los eslabones y la posición inicial de las juntas

In [2]:
from sympy import *

#Matrices de rotación
def r_z(alpha):
  rz = Matrix([[cos(alpha), -sin(alpha), 0], 
               [sin(alpha),  cos(alpha), 0], 
               [0         ,           0, 1]])
  return rz



th1, th2, th3 = symbols("theta_1, theta_2, theta_3")
r_z(th3)

#para un manipulador de 3GDL en el plano XY
th1, th2, th3 = symbols("theta_1, theta_2, theta_3")
l1, l2, l3 = symbols("l_1, l_2, l_3")
p_0_1 = Matrix([[0],[0],[0]])
p_1_2 = Matrix([[l1],[0],[0]])
p_2_3 = Matrix([[l2],[0],[0]])
p_3_p = Matrix([[l3],[0],[0]])
R_0_1 = r_z(th1)
R_1_2 = r_z(th2)
R_2_3 = r_z(th3)
R_3_p = r_z(0)


def tr_h(rot, pos):
  a = Matrix.vstack(rot, Matrix([[0,0,0]]))
  b = Matrix.vstack(pos, Matrix([1]))
  t_h = Matrix.hstack(a, b)
  return t_h

T_0_1 = tr_h(R_0_1, p_0_1)
T_1_2 = tr_h(R_1_2, p_1_2)
T_2_3 = tr_h(R_2_3, p_2_3)
T_3_p = tr_h(R_3_p, p_3_p)
T_0_p = T_0_1 * T_1_2 * T_2_3 * T_3_p
T_0_p = simplify(T_0_p)



simplify(T_0_p)

xi = Matrix([T_0_p[0, 3],
             T_0_p[1, 3],
             atan2(T_0_p[1, 0], T_0_p[0,0])])
xi = xi.subs({l1: 0.1, l2: 0.1, l3: 0.1})
simplify(xi)



xi.subs({th1: 0, th2: 0, th3: 0})

J = Matrix([[diff(xi, th1), 
             diff(xi, th2), 
             diff(xi, th3)]])
J = simplify(J)
J_inv = simplify(J.inverse())
J_inv

Matrix([
[                       10.0*cos(theta_1 + theta_2)/sin(theta_2),                        10.0*sin(theta_1 + theta_2)/sin(theta_2),                                 1.0*sin(theta_3)/sin(theta_2)],
[-(10.0*cos(theta_1) + 10.0*cos(theta_1 + theta_2))/sin(theta_2), -(10.0*sin(theta_1) + 10.0*sin(theta_1 + theta_2))/sin(theta_2), -(1.0*sin(theta_3) + 1.0*sin(theta_2 + theta_3))/sin(theta_2)],
[                                 10.0*cos(theta_1)/sin(theta_2),                                  10.0*sin(theta_1)/sin(theta_2),        1.0*sin(theta_3)/tan(theta_2) + 1.0*cos(theta_3) + 1.0]])

### 2. Planteamiento de la trayectoria

En esta segunda parte, se planteará el código que permita definir los puntos intermedios de una trayectoria, la cual debe tener velocidades y aceleraciones nulas al inicio y al final. Se deben incluir también las gráficas de la posición, velocidad y aceleración del efector final. 

Calcular la trayectoria considerando de forma general tiempo de duración, puntos inicial y final, y con una tasa de muestreo de 30 muestras por segundo. 

In [3]:
t = symbols("t")

a0,a1,a2,a3,a4,a5 = symbols(
    "a0 a1 a2 a3 a4 a5"
)

lam = a0+a1*t+a2*t**2+a3*t**3+a4*t**4+a5*t**5

lam_dot = diff(lam,t)
lam_dot_dot = diff(lam_dot,t)

tf = 2

eq1 = lam.subs(t,0)
eq2 = lam.subs(t,tf)-1

eq3 = lam_dot.subs(t,0)
eq4 = lam_dot.subs(t,tf)

eq5 = lam_dot_dot.subs(t,0)
eq6 = lam_dot_dot.subs(t,tf)

solutions = solve(
    (eq1,eq2,eq3,eq4,eq5,eq6),
    (a0,a1,a2,a3,a4,a5)
)

lam_s = lam.subs(solutions)

display(lam_s)

th1_in = 0.1
th2_in = 0.1
th3_in = 0.1

x_in = xi[0].subs({
    th1:th1_in,
    th2:th2_in,
    th3:th3_in
})

y_in = xi[1].subs({
    th1:th1_in,
    th2:th2_in,
    th3:th3_in
})

alpha_in = xi[2].subs({
    th1:th1_in,
    th2:th2_in,
    th3:th3_in
})

x_f = 0.2
y_f = 0.1
alpha_f = pi/12
x_eq = x_in + lam_s*(x_f-x_in)
y_eq = y_in + lam_s*(y_f-y_in)
alpha_eq = alpha_in + lam_s*(alpha_f-alpha_in)

x_dot_eq = diff(x_eq,t)
y_dot_eq = diff(y_eq,t)
alpha_dot_eq = diff(alpha_eq,t)

x_dot_dot_eq = diff(x_dot_eq,t)
y_dot_dot_eq = diff(y_dot_eq,t)
alpha_dot_dot_eq = diff(alpha_dot_eq,t)

3*t**5/16 - 15*t**4/16 + 5*t**3/4

### 3. Cinemática inversa
A partir del modelo de la cinemática directa, obtener la expresión e la cinemática inversa, que relacione las velocidades de las juntas del robot con la velocidad del efector final. Ya que el modelo de cinemática inversa sólo permite obtener velocidades, obtener también expresiones que permitan obtener la posición de las juntas y sus aceleraciones

In [4]:
J_inv = simplify(J.inv())

display(J_inv)
x_dot,y_dot,alpha_dot = symbols(
    "x_dot y_dot alpha_dot"
)

xi_dot = Matrix([
    x_dot,
    y_dot,
    alpha_dot
])

th_dot = simplify(J_inv*xi_dot)

display(th_dot)

Matrix([
[                       10.0*cos(theta_1 + theta_2)/sin(theta_2),                        10.0*sin(theta_1 + theta_2)/sin(theta_2),                                 1.0*sin(theta_3)/sin(theta_2)],
[-(10.0*cos(theta_1) + 10.0*cos(theta_1 + theta_2))/sin(theta_2), -(10.0*sin(theta_1) + 10.0*sin(theta_1 + theta_2))/sin(theta_2), -(1.0*sin(theta_3) + 1.0*sin(theta_2 + theta_3))/sin(theta_2)],
[                                 10.0*cos(theta_1)/sin(theta_2),                                  10.0*sin(theta_1)/sin(theta_2),        1.0*sin(theta_3)/tan(theta_2) + 1.0*cos(theta_3) + 1.0]])

Matrix([
[                                                              (1.0*alpha_dot*sin(theta_3) + 10.0*x_dot*cos(theta_1 + theta_2) + 10.0*y_dot*sin(theta_1 + theta_2))/sin(theta_2)],
[(-1.0*alpha_dot*(sin(theta_3) + sin(theta_2 + theta_3)) - 10.0*x_dot*(cos(theta_1) + cos(theta_1 + theta_2)) - 10.0*y_dot*(sin(theta_1) + sin(theta_1 + theta_2)))/sin(theta_2)],
[             1.0*alpha_dot*sin(theta_3)/tan(theta_2) + 1.0*alpha_dot*cos(theta_3) + 1.0*alpha_dot + 10.0*x_dot*cos(theta_1)/sin(theta_2) + 10.0*y_dot*sin(theta_1)/sin(theta_2)]])

### 4. Aplicación de la cinemática inversa
Finalmente, a partir de los puntos de la trayectoria y el modelo de cinemática inversa, obtener las posiciones, velocidades y aceleraciones de las juntas del robot, así como sus gráficas en función del tiempo

In [5]:
frec = 30
dt = 1/frec

samples = int(frec*tf)+1

### 5. Repositorio
Para terminar, subir los archivos de la práctica al repositorio de github

## Análisis de resultados



- ¿Qué utilidad tiene el modelo de cinemática inversa de un robot?
> Permite determinar los movimientos o ángulos de las articulaciones necesarios para que el robot alcance una posición y orientación específica.


## Conclusiones



> Respuesta



## Bibliografía 



> En caso de usarse, se deben hacer referencia a la información implementada en formato ieee


